# Week 2 — concept map in code

One cell of theory, then one standalone snippet per concept. Generic data, no dependency on the triage project.
Section numbers match `week-02.md`. Cells marked **[$]** call the paid API (fractions of a cent on Haiku 5.5); **[free]** cells make no billed call; unmarked cells are pure Python.

| § | Concepts |
|---|---|
| 0 | Setup, helpers |
| 1 | Context limits, price tiers, `count_tokens` gate, normalization, collapse + rare rule, source pointers, chunking, map-reduce |
| 2 | Statelessness, prefill, `cache_control` (block, system, top-level, 1h), three counters, minimum length, invalidation tiers, silent invalidators, break-even, multi-turn caching, pre-warm, response cache |
| 3 | Error classes, retryable vs not, client config, backoff + jitter, `retry-after`, retry budgets, `stop_reason`, validation-retry, exit codes |
| 4 | Versioned prompts, hashing, immutability, A/B harness, version in cache keys |
| 5 | Batches: `custom_id`, create, poll, results, routing, structured output in batch, caching in batch, cancel/list |
| 6 | JSONL logging, aggregation, hit rate, cost per completed task |

## 0. Setup

In [1]:
import os, re, json, time, random, hashlib, uuid
from collections import Counter
from pathlib import Path
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))                                # ANTHROPIC_API_KEY from .env
client = anthropic.Anthropic(timeout=60.0, max_retries=4)
MODEL = "claude-haiku-5-5"
print("SDK", anthropic.__version__)

SDK 2.6.0


In [2]:
def text_of(msg) -> str:
    """Concatenate text blocks only. content[0] may be a thinking block (week 1 weak spot)."""
    return "".join(b.text for b in msg.content if b.type == "text")

def usage_line(label, u):
    w, r, i = u.cache_creation_input_tokens or 0, u.cache_read_input_tokens or 0, u.input_tokens
    print(f"{label:<34} write={w:>6} read={r:>6} input={i:>6} total_prompt={w + r + i:>7} out={u.output_tokens}")

# A deterministic ~1,500-token reference document, used wherever a cacheable prefix (>= 512 tokens) is needed
REFERENCE = "\n".join(
    f"Rule {i:03d}: when service-{i % 7} reports error E{i % 13} for more than {i % 5 + 1} minutes, "
    f"page team-{i % 4} and attach the last {10 * (i % 3 + 1)} log lines."
    for i in range(1, 61))
print(len(REFERENCE), "chars")

6791 chars


## 1. Long inputs

### 1.1 Three limits: window (hard), price (soft), quality (silent)
The context window rejects; price tiers charge more past a threshold; quality degrades with noise and never errors. Encode the first two as data so code can decide.

In [3]:
LIMITS = {   # model: (context window, long-prompt threshold, $/M in, $/M out, $/M in long, $/M out long)
    "claude-haiku-5-5": (1_000_000, 100_000, 0.10, 0.50, 0.50, 2.50),
}

def price(model: str, prompt_tokens: int, output_tokens: int) -> float:
    window, tier, pin, pout, pin_l, pout_l = LIMITS[model]
    if prompt_tokens > window:
        raise ValueError(f"{prompt_tokens:,} tokens > {window:,} window: request would be rejected")
    if prompt_tokens > tier:
        pin, pout = pin_l, pout_l
    return (prompt_tokens * pin + output_tokens * pout) / 1e6

for n in (2_000, 99_000, 101_000, 900_000):
    print(f"{n:>9,} prompt tokens -> ${price(MODEL, n, 700):.5f}")
try:
    price(MODEL, 2_400_000, 700)
except ValueError as e:
    print(e)

    2,000 prompt tokens -> $0.00055
   99,000 prompt tokens -> $0.01025
  101,000 prompt tokens -> $0.05225
  900,000 prompt tokens -> $0.45175
2,400,000 tokens > 1,000,000 window: request would be rejected


### 1.2 Measure first: `count_tokens` as a gate  [free]
Same body as a Messages request, returns `input_tokens`, generates nothing. Pass the same `system`, `messages`, tools and output format you will send: all of them count.

In [4]:
def prompt_tokens(system: str, messages: list, **extra) -> int:
    return client.messages.count_tokens(model=MODEL, system=system, messages=messages, **extra).input_tokens

def choose_strategy(n: int, budget: int = 100_000) -> str:
    window = LIMITS[MODEL][0]
    return "reject / map-reduce" if n > window else "reduce first" if n > budget else "send as is"

doc = REFERENCE * 3
n = prompt_tokens("You answer questions about the rules.", [{"role": "user", "content": doc}])
print(f"{len(doc):,} chars -> {n:,} tokens ({len(doc) / n:.1f} chars/token) -> {choose_strategy(n)}")

20,373 chars -> 8,116 tokens (2.5 chars/token) -> send as is


In [5]:
# Structured output adds the schema to the prompt: count it too
from pydantic import BaseModel
from anthropic import transform_schema

class Answer(BaseModel):
    rule_numbers: list[int]
    reasoning: str

msgs = [{"role": "user", "content": doc + "\n\nWhich rules page team-2?"}]
plain = prompt_tokens("Answer from the rules.", msgs)
with_schema = prompt_tokens("Answer from the rules.", msgs,
                            output_config={"format": {"type": "json_schema", "schema": transform_schema(Answer)}})
print(f"plain {plain:,} | with schema {with_schema:,} | schema overhead {with_schema - plain:,} tokens")

plain 8,124 | with schema 8,401 | schema overhead 277 tokens


### 1.3 Normalization: same event, different variables → one signature
Replace variable parts with placeholders. Order matters: specific patterns (uuid, hex, ip) before generic numbers. A missing pattern means no collapse (the `\\b\\d+\\b` vs `479ms` bug).

In [6]:
VARS = [
    (re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b"), "<uuid>"),
    (re.compile(r"\b\d{1,3}(\.\d{1,3}){3}\b"), "<ip>"),
    (re.compile(r"\b[0-9a-f]{7,40}\b"), "<hex>"),
    (re.compile(r"\d+(\.\d+)?"), "<n>"),          # no \b: must match the 479 in "479ms"
]

def normalize(msg: str) -> str:
    for rx, rep in VARS:
        msg = rx.sub(rep, msg)
    return msg

for m in ["timeout after 479ms from 10.0.3.17",
          "timeout after 12ms from 10.0.9.2",
          "deploy 7f3a91c rolled out to 40 pods",
          "req 3f2a9c1e-5b7d-4e8a-9c21-7d4e5f6a8b90 failed"]:
    print(f"{m:<52} -> {normalize(m)}")

broken = re.compile(r"\b\d+\b")
print("\nbuggy pattern:", broken.sub("<n>", "timeout after 479ms"))

timeout after 479ms from 10.0.3.17                   -> timeout after <n>ms from <ip>
timeout after 12ms from 10.0.9.2                     -> timeout after <n>ms from <ip>
deploy 7f3a91c rolled out to 40 pods                 -> deploy <hex> rolled out to <n> pods
req 3f2a9c1e-5b7d-4e8a-9c21-7d4e5f6a8b90 failed      -> req <uuid> failed

buggy pattern: timeout after 479ms


### 1.4 Collapse + keep rule: frequent background vs rare change
Group by signature, keep count + first occurrence verbatim. Keep everything severe **and** everything rare: a one-off change is rare whatever its level.

In [7]:
random.seed(7)
log = []
for i in range(5_000):
    if i == 2_400:
        log.append("INFO  [config] max_workers changed 32 -> 2")            # the cause, once
    elif i > 2_405 and random.random() < 0.2:
        log.append(f"ERROR [queue] job {random.randint(1, 9999)} timed out after {random.randint(100, 999)}ms")
    else:
        log.append(random.choice([f"INFO  [http] GET /health 200 {random.randint(1, 50)}ms",
                                  f"WARN  [gc] pause {random.randint(5, 80)}ms",
                                  f"INFO  [jobs] job {random.randint(1, 9999)} done"]))

def collapse(lines, rare=3, keep_levels=("WARN", "ERROR")):
    groups = {}
    for no, line in enumerate(lines, start=1):
        level = line.split()[0]
        g = groups.setdefault(normalize(line), {"first": no, "text": line, "level": level, "n": 0})
        g["n"] += 1
    kept = [g for g in groups.values() if g["level"] in keep_levels or g["n"] <= rare]
    return sorted(kept, key=lambda g: g["first"])

for g in collapse(log):
    print(f'{g["first"]:>5}  {g["n"]:>5}x  {g["text"]}')
print("\nlevel-only filter keeps the cause?",
      any("config" in g["text"] for g in collapse(log, rare=0)))

    4   1552x  WARN  [gc] pause 9ms
 2401      1x  INFO  [config] max_workers changed 32 -> 2
 2409    511x  ERROR [queue] job 7609 timed out after 420ms

level-only filter keeps the cause? False


### 1.5 Keep a pointer back to the source
The model sees the reduced view; every claim must be checkable against the original. Here: line numbers + verbatim text, verified by code.

In [8]:
def verify(citations: list[tuple[int, str]], source: list[str]) -> list[str]:
    problems = []
    for no, text in citations:
        if not 1 <= no <= len(source):
            problems.append(f"line {no}: out of range")
        elif source[no - 1] != text:
            problems.append(f"line {no}: text mismatch")
    return problems

print(verify([(2401, log[2400]), (2401, "INFO  [config] max_workers changed 32 -> 4"), (9999, "x")], log))

['line 2401: text mismatch', 'line 9999: out of range']


### 1.6 Chunking by token budget
Map-reduce needs chunks that each fit a budget. Split on natural boundaries (lines), never mid-record.

In [9]:
def chunk_lines(lines: list[str], max_chars: int) -> list[list[str]]:
    chunks, cur, size = [], [], 0
    for line in lines:
        if size + len(line) > max_chars and cur:
            chunks.append(cur); cur, size = [], 0
        cur.append(line); size += len(line) + 1
    if cur:
        chunks.append(cur)
    return chunks

chunks = chunk_lines(log, max_chars=40_000)
print(len(chunks), "chunks:", [len(c) for c in chunks])

4 chunks: [1445, 1398, 1300, 857]


### 1.7 Map-reduce  [$]
Map: one cheap structured extraction per chunk. Reduce: one call over all extractions. Costs N+1 calls; evidence split across chunks is never seen together.

In [10]:
class ChunkFindings(BaseModel):
    candidate_causes: list[str]          # changes that could explain errors
    error_signatures: list[str]

class Verdict(BaseModel):
    root_cause: str
    confidence: float

def map_chunk(lines: list[str]) -> ChunkFindings:
    r = client.messages.parse(model=MODEL, max_tokens=2000, output_config={"effort": "low"},
                              output_format=ChunkFindings,
                              messages=[{"role": "user", "content":
                                         "<logs>\n" + "\n".join(lines) + "\n</logs>\nExtract candidate causes and error signatures."}])
    return r.parsed_output

def reduce_findings(findings: list[ChunkFindings]) -> Verdict:
    r = client.messages.parse(model=MODEL, max_tokens=2000, output_config={"effort": "low"},
                              output_format=Verdict,
                              messages=[{"role": "user", "content":
                                         "Per-chunk findings, in time order:\n"
                                         + "\n".join(f.model_dump_json() for f in findings)
                                         + "\nWhat is the root cause?"}])
    return r.parsed_output

findings = [map_chunk(c) for c in chunks]     # N calls
print(reduce_findings(findings))              # +1 call

root_cause='The max_workers setting was reduced from 32 to 2 via a configuration change, cutting worker capacity so the job queue backs up; queued jobs then exceed the timeout and fail with ERROR [queue] job timed out. Frequent GC pauses (often 40-80ms) compound the latency but are a secondary contributor rather than the primary cause.' confidence=0.8


## 2. Prompt caching

### 2.1 The API is stateless: every turn resends everything  [$]
Watch `input_tokens` grow turn over turn without caching. This is the cost caching removes.

In [11]:
history = [{"role": "user", "content": REFERENCE + "\n\nWhich team handles service-3?"}]
for turn in range(3):
    r = client.messages.create(model=MODEL, max_tokens=300, output_config={"effort": "low"}, messages=history)
    usage_line(f"turn {turn + 1} (no caching)", r.usage)
    history += [{"role": "assistant", "content": r.content},
                {"role": "user", "content": "And for the next service number?"}]

turn 1 (no caching)                write=     0 read=     0 input=  2717 total_prompt=   2717 out=300
turn 2 (no caching)                write=     0 read=     0 input=  3030 total_prompt=   3030 out=300
turn 3 (no caching)                write=     0 read=     0 input=  3344 total_prompt=   3344 out=300


### 2.2 Block-level marker  [$]
`cache_control` on a content block = cache the prefix up to and including that block. Content must be a list of blocks. Call twice: write, then read.

In [12]:
def ask(question: str, marker=True, system=None):
    block = {"type": "text", "text": REFERENCE}
    if marker:
        block["cache_control"] = {"type": "ephemeral"}            # default TTL 5 minutes
    kw = dict(model=MODEL, max_tokens=200, output_config={"effort": "low"},
              messages=[{"role": "user", "content": [block, {"type": "text", "text": question}]}])
    if system:
        kw["system"] = system
    return client.messages.create(**kw)

usage_line("call 1", ask("Which team handles service-3?").usage)
usage_line("call 2, different question", ask("Which rules mention E5?").usage)   # question is after the marker

call 1                             write=  2703 read=     0 input=    13 total_prompt=   2716 out=200
call 2, different question         write=     0 read=  2703 input=    13 total_prompt=   2716 out=200


### 2.3 Marker on the system prompt; top-level automatic caching; 1-hour TTL  [$]

In [13]:
# (a) system as a list of blocks, marker on the system block
r = client.messages.create(model=MODEL, max_tokens=100, output_config={"effort": "low"},
                           system=[{"type": "text", "text": "You answer from these rules.\n" + REFERENCE,
                                    "cache_control": {"type": "ephemeral"}}],
                           messages=[{"role": "user", "content": "Which team handles service-1?"}])
usage_line("(a) system marker", r.usage)

# (b) top-level: the server places the marker on the last cacheable block (good for chats)
r = client.messages.create(model=MODEL, max_tokens=100, output_config={"effort": "low"},
                           cache_control={"type": "ephemeral"},
                           messages=[{"role": "user", "content": REFERENCE + "\nWhich team handles service-2?"}])
usage_line("(b) automatic", r.usage)

# (c) 1-hour TTL: 2x write price, survives gaps of 5-60 minutes
r = client.messages.create(model=MODEL, max_tokens=100, output_config={"effort": "low"},
                           messages=[{"role": "user", "content": [
                               {"type": "text", "text": REFERENCE, "cache_control": {"type": "ephemeral", "ttl": "1h"}},
                               {"type": "text", "text": "Which team handles service-4?"}]}])
usage_line("(c) 1h TTL", r.usage)
print(getattr(r.usage, "cache_creation", None))           # breakdown by TTL when present

(a) system marker                  write=  2711 read=     0 input=    15 total_prompt=   2726 out=100
(b) automatic                      write=  2713 read=     0 input=     4 total_prompt=   2717 out=100
(c) 1h TTL                         write=     0 read=  2703 input=    13 total_prompt=   2716 out=100
CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0)


### 2.4 The three counters sum to the prompt; each has its own price
`input_tokens` alone is only the uncached tail.

In [14]:
def prompt_size(u) -> int:
    return u.input_tokens + (u.cache_creation_input_tokens or 0) + (u.cache_read_input_tokens or 0)

def cost_with_cache(u, pin=0.10, pout=0.50, ttl="5m") -> float:
    write_mult = 1.25 if ttl == "5m" else 2.0
    return (u.input_tokens * pin
            + (u.cache_creation_input_tokens or 0) * pin * write_mult
            + (u.cache_read_input_tokens or 0) * pin * 0.10
            + u.output_tokens * pout) / 1e6

from types import SimpleNamespace as U
miss = U(input_tokens=12, cache_creation_input_tokens=90_000, cache_read_input_tokens=0, output_tokens=600)
hit  = U(input_tokens=12, cache_creation_input_tokens=0, cache_read_input_tokens=90_000, output_tokens=600)
for name, u in (("miss", miss), ("hit", hit)):
    print(f"{name}: prompt={prompt_size(u):,}  cost=${cost_with_cache(u):.6f}")

miss: prompt=90,012  cost=$0.011551
hit: prompt=90,012  cost=$0.001201


### 2.5 Below the minimum: silently not cached  [$]
512 tokens on the 5.5 models. No error; both counters stay 0.

In [15]:
for n in (1, 2):
    r = client.messages.create(model=MODEL, max_tokens=20, output_config={"effort": "low"},
                               messages=[{"role": "user", "content": [
                                   {"type": "text", "text": "Short context.", "cache_control": {"type": "ephemeral"}},
                                   {"type": "text", "text": "Say OK."}]}])
    usage_line(f"tiny prefix call {n}", r.usage)

tiny prefix call 1                 write=     0 read=     0 input=    17 total_prompt=     17 out=5
tiny prefix call 2                 write=     0 read=     0 input=    17 total_prompt=     17 out=5


### 2.6 Invalidation tiers  [$]
Prefix order is tools → system → messages. A change invalidates its tier and everything after it. Predict each line before running.

In [16]:
TOOLS_A = [{"name": "lookup_rule", "description": "Look up a rule by number.",
            "input_schema": {"type": "object", "properties": {"n": {"type": "integer"}}, "required": ["n"]}}]
TOOLS_B = TOOLS_A + [{"name": "page_team", "description": "Page a team.",
                      "input_schema": {"type": "object", "properties": {"team": {"type": "string"}}, "required": ["team"]}}]
SYS = "You answer from the rules."

def tiered(tools, system, question):
    return client.messages.create(
        model=MODEL, max_tokens=100, output_config={"effort": "low"}, tools=tools,
        system=[{"type": "text", "text": system}],
        messages=[{"role": "user", "content": [
            {"type": "text", "text": REFERENCE, "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": question}]}])

usage_line("baseline", tiered(TOOLS_A, SYS, "Rule 7?").usage)
usage_line("same again", tiered(TOOLS_A, SYS, "Rule 7?").usage)
usage_line("question changed (after marker)", tiered(TOOLS_A, SYS, "Rule 9?").usage)
usage_line("system changed", tiered(TOOLS_A, SYS + " Be brief.", "Rule 7?").usage)
usage_line("tool added", tiered(TOOLS_B, SYS, "Rule 7?").usage)

baseline                           write=  3072 read=     0 input=    10 total_prompt=   3082 out=100
same again                         write=     0 read=  3072 input=    10 total_prompt=   3082 out=100
question changed (after marker)    write=     0 read=  3072 input=    10 total_prompt=   3082 out=100
system changed                     write=  3077 read=     0 input=    10 total_prompt=   3087 out=100
tool added                         write=  3138 read=     0 input=    10 total_prompt=   3148 out=100


### 2.7 Silent invalidators: make the prefix deterministic
Anything that changes bytes before the marker zeroes the hit rate. Compare fingerprints of what you would send.

In [17]:
from datetime import datetime

def fingerprint(obj) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:10]

bad_system = lambda: f"You are a helper. Current time: {datetime.now().isoformat()}"
good_system = lambda: "You are a helper."
print("timestamp in system:", fingerprint(bad_system()), fingerprint(bad_system()))
print("static system:      ", fingerprint(good_system()), fingerprint(good_system()))

cfg = {"b": 2, "a": 1}
print("json without sort_keys:", json.dumps(cfg), "| with:", json.dumps(cfg, sort_keys=True))
tags = {"db", "cache", "auth"}
print("set iteration order is not a contract:", ", ".join(tags), "| use sorted():", ", ".join(sorted(tags)))

timestamp in system: 6f14f6ec04 fc680340c4
static system:       02e796b99c 02e796b99c
json without sort_keys: {"b": 2, "a": 1} | with: {"a": 1, "b": 2}
set iteration order is not a contract: db, auth, cache | use sorted(): auth, cache, db


### 2.8 Break-even: when does caching pay?
Writes cost 1.25x (5m) or 2x (1h); reads ~0.1x. N calls on the same prefix.

In [18]:
def relative_cost(n_calls: int, ttl: str = "5m", read: float = 0.10) -> float:
    write = 1.25 if ttl == "5m" else 2.0
    return (write + (n_calls - 1) * read) / n_calls          # vs 1.0 uncached per call

for n in (1, 2, 3, 5, 10):
    print(f"{n:>2} calls: 5m -> {relative_cost(n):.2f}x   1h -> {relative_cost(n, '1h'):.2f}x  of uncached")

 1 calls: 5m -> 1.25x   1h -> 2.00x  of uncached
 2 calls: 5m -> 0.68x   1h -> 1.05x  of uncached
 3 calls: 5m -> 0.48x   1h -> 0.73x  of uncached
 5 calls: 5m -> 0.33x   1h -> 0.48x  of uncached
10 calls: 5m -> 0.21x   1h -> 0.29x  of uncached


### 2.9 Multi-turn / agent loop: explicit marker on the static part + automatic for the growing tail  [$]
Append-only history; never edit earlier turns. `read` should grow turn over turn while `write` stays small.

In [19]:
static_system = [{"type": "text", "text": "You answer from these rules.\n" + REFERENCE,
                  "cache_control": {"type": "ephemeral"}}]               # explicit: static part
chat = [{"role": "user", "content": "Which team handles service-3?"}]
for turn in range(3):
    r = client.messages.create(model=MODEL, max_tokens=200, output_config={"effort": "low"},
                               system=static_system,
                               cache_control={"type": "ephemeral"},         # automatic: growing tail
                               messages=chat)
    usage_line(f"turn {turn + 1}", r.usage)
    chat += [{"role": "assistant", "content": r.content},                 # append, never rewrite
             {"role": "user", "content": "And the next service?"}]

turn 1                             write=    11 read=  2711 input=     4 total_prompt=   2726 out=200
turn 2                             write=   212 read=  2722 input=     4 total_prompt=   2938 out=200
turn 3                             write=   212 read=  2934 input=     4 total_prompt=   3150 out=200


### 2.10 Pre-warm with `max_tokens=0`  [$]
Runs prefill only, writes the cache, returns `content: []`. The first real request is then a hit.

In [20]:
warm = client.messages.create(model=MODEL, max_tokens=0,
                              messages=[{"role": "user", "content": [
                                  {"type": "text", "text": REFERENCE + "\nWarm.", "cache_control": {"type": "ephemeral"}}]}])
usage_line("pre-warm", warm.usage)
print("content:", warm.content, "| stop_reason:", warm.stop_reason)

pre-warm                           write=  2708 read=     0 input=     4 total_prompt=   2712 out=0
content: [] | stop_reason: max_tokens


### 2.11 Response cache (application side)
Skips the call entirely. Exact-match key over everything that shapes the output, tenant-scoped, validated results only, TTL.

In [ ]:
class ResponseCache:
    def __init__(self, ttl_s: int = 86_400):
        self.store, self.ttl = {}, ttl_s

    @staticmethod
    def key(tenant, model, effort, prompt_version, system, messages) -> str:
        blob = json.dumps({"t": tenant, "m": model, "e": effort, "p": prompt_version,
                           "s": system, "msg": messages}, sort_keys=True)
        return hashlib.sha256(blob.encode()).hexdigest()

    def get(self, k):
        hit = self.store.get(k)
        if hit and time.time() - hit[0] < self.ttl:
            return hit[1]
        return None

    def put(self, k, value, valid: bool):
        if valid:                                   # never cache an answer that failed validation
            self.store[k] = (time.time(), value)

rc = ResponseCache()
k_a = rc.key("tenant-a", MODEL, "low", "qa.v1", "sys", [{"role": "user", "content": "q"}])
k_b = rc.key("tenant-b", MODEL, "low", "qa.v1", "sys", [{"role": "user", "content": "q"}])
k_v2 = rc.key("tenant-a", MODEL, "low", "qa.v2", "sys", [{"role": "user", "content": "q"}])
rc.put(k_a, "answer", valid=True)
print("same tenant:", rc.get(k_a), "| other tenant:", rc.get(k_b), "| new prompt version:", rc.get(k_v2))

## 3. Failure handling

### 3.1 The exception hierarchy
Catch most-specific first. `OverloadedError` (529) is a subclass of `InternalServerError`; `APIConnectionError` is a sibling of `APIStatusError` in Python.

In [21]:
for cls in (anthropic.BadRequestError, anthropic.AuthenticationError, anthropic.PermissionDeniedError,
            anthropic.NotFoundError, anthropic.RateLimitError, anthropic.InternalServerError,
            anthropic.OverloadedError, anthropic.APITimeoutError, anthropic.APIConnectionError):
    print(f"{cls.__name__:<22} <- " + " <- ".join(c.__name__ for c in cls.__mro__[1:5]))

BadRequestError        <- APIStatusError <- APIError <- AnthropicError <- Exception
AuthenticationError    <- APIStatusError <- APIError <- AnthropicError <- Exception
PermissionDeniedError  <- APIStatusError <- APIError <- AnthropicError <- Exception
NotFoundError          <- APIStatusError <- APIError <- AnthropicError <- Exception
RateLimitError         <- APIStatusError <- APIError <- AnthropicError <- Exception
InternalServerError    <- APIStatusError <- APIError <- AnthropicError <- Exception
OverloadedError        <- InternalServerError <- APIStatusError <- APIError <- AnthropicError
APITimeoutError        <- APIConnectionError <- APIError <- AnthropicError <- Exception
APIConnectionError     <- APIError <- AnthropicError <- Exception <- BaseException


### 3.2 Retryable or not
4xx: your request is wrong, the same bytes fail again. 429 / 5xx / network: temporary.

In [22]:
def retryable(e: Exception) -> bool:
    if isinstance(e, (anthropic.APIConnectionError, anthropic.APITimeoutError)):
        return True
    if isinstance(e, anthropic.APIStatusError):
        return e.status_code == 429 or e.status_code >= 500
    return False

class FakeStatus(anthropic.APIStatusError):
    def __init__(self, code):
        self.status_code = code
for code in (400, 401, 404, 413, 429, 500, 529):
    print(code, "retry" if retryable(FakeStatus(code)) else "do not retry")

400 do not retry
401 do not retry
404 do not retry
413 do not retry
429 retry
500 retry
529 retry


### 3.3 Client configuration: what the SDK already does
Default `max_retries=2` (with backoff, honoring `retry-after`), `timeout=600 s` per attempt. Worst case = timeout × (retries + 1).

In [23]:
default = anthropic.Anthropic()
tuned = anthropic.Anthropic(timeout=60.0, max_retries=4)
granular = anthropic.Anthropic(timeout=anthropic.Timeout(60.0, connect=2.0, read=30.0, write=10.0))
for name, c in (("default", default), ("tuned", tuned), ("granular", granular)):
    print(f"{name:<9} timeout={c.timeout}  max_retries={c.max_retries}")

# per-call override without mutating the shared client
fast_fail = client.with_options(timeout=10.0, max_retries=0)
print("override:", fast_fail.timeout, fast_fail.max_retries)

default   timeout=Timeout(connect=5.0, read=600, write=600, pool=600)  max_retries=2
tuned     timeout=60.0  max_retries=4
granular  timeout=Timeout(connect=2.0, read=30.0, write=10.0, pool=60.0)  max_retries=2
override: 10.0 0


### 3.4 Exponential backoff with full jitter (what the SDK does for you)
Delay doubles per attempt, capped, randomized. Jitter prevents a thundering herd of clients retrying in lockstep.

In [24]:
def backoff_delays(attempts: int, base: float = 0.5, cap: float = 8.0, seed: int = 0) -> list[float]:
    rnd = random.Random(seed)
    return [round(rnd.uniform(0, min(cap, base * 2 ** i)), 2) for i in range(attempts)]

print("no jitter :", [min(8.0, 0.5 * 2 ** i) for i in range(6)])
for s in range(3):
    print(f"client {s}  :", backoff_delays(6, seed=s))

no jitter : [0.5, 1.0, 2.0, 4.0, 8.0, 8.0]
client 0  : [0.42, 0.76, 0.84, 1.04, 4.09, 3.24]
client 1  : [0.07, 0.85, 1.53, 1.02, 3.96, 3.6]
client 2  : [0.48, 0.95, 0.11, 0.34, 6.68, 5.89]


In [25]:
# Manual retry loop with the SDK's own retries switched off: same idea, visible
def with_retries(fn, attempts=4):
    no_retry = client.with_options(max_retries=0)
    for i in range(attempts + 1):
        try:
            return fn(no_retry)
        except anthropic.APIError as e:
            if not retryable(e) or i == attempts:
                raise
            delay = float(getattr(getattr(e, "response", None), "headers", {}).get("retry-after", 0) or 0) \
                    or backoff_delays(attempts, seed=i)[i]
            print(f"attempt {i + 1} failed ({type(e).__name__}), sleeping {delay:.2f}s")
            time.sleep(delay)

### 3.5 Reading `retry-after` and rate-limit headers  [$]
Every response carries rate-limit headers; a 429 carries `retry-after`. `with_raw_response` exposes headers on success.

In [26]:
raw = client.messages.with_raw_response.create(model=MODEL, max_tokens=10, output_config={"effort": "low"},
                                               messages=[{"role": "user", "content": "Say OK."}])
for h, v in raw.headers.items():
    if h.startswith("anthropic-ratelimit") or h in ("retry-after", "request-id"):
        print(f"{h}: {v}")
msg = raw.parse()                                     # the normal Message object
print(text_of(msg))

anthropic-ratelimit-input-tokens-limit: 10000000
anthropic-ratelimit-input-tokens-remaining: 10000000
anthropic-ratelimit-input-tokens-reset: 2026-11-07T09:08:14Z
anthropic-ratelimit-output-tokens-limit: 2000000
anthropic-ratelimit-output-tokens-remaining: 2000000
anthropic-ratelimit-output-tokens-reset: 2026-11-07T09:08:14Z
anthropic-ratelimit-requests-limit: 13000
anthropic-ratelimit-requests-remaining: 12999
anthropic-ratelimit-requests-reset: 2026-11-07T09:08:13Z
anthropic-ratelimit-tokens-limit: 12000000
anthropic-ratelimit-tokens-remaining: 12000000
anthropic-ratelimit-tokens-reset: 2026-11-07T09:08:14Z
request-id: req_011CgoJcj53crryManuWtykD
OK


### 3.6 Retry budgets multiply
Every retry layer multiplies requests and wall-clock. Compute the worst case before shipping.

In [27]:
def worst_case(timeout_s, sdk_retries, validation_retries=1, agent_steps=1, backoff_cap=8.0):
    attempts = sdk_retries + 1
    logical = (validation_retries + 1) * agent_steps
    requests = logical * attempts
    backoff = logical * sum(min(backoff_cap, 0.5 * 2 ** i) for i in range(sdk_retries))
    return requests, (requests * timeout_s + backoff) / 60

for cfg in [(600, 2, 1, 1), (60, 4, 1, 1), (60, 4, 1, 15)]:
    req, minutes = worst_case(*cfg)
    print(f"timeout={cfg[0]}s retries={cfg[1]} validation={cfg[2]} steps={cfg[3]:>2} -> "
          f"{req:>3} requests, {minutes:6.1f} min worst case")

timeout=600s retries=2 validation=1 steps= 1 ->   6 requests,   60.0 min worst case
timeout=60s retries=4 validation=1 steps= 1 ->  10 requests,   10.2 min worst case
timeout=60s retries=4 validation=1 steps=15 -> 150 requests,  153.8 min worst case


### 3.7 Family B: HTTP 200 but unusable — check `stop_reason` first  [$]
`max_tokens` = truncated (partial JSON); `refusal` = declined. Never use the content in either case.

In [28]:
class Truncated(RuntimeError): pass

def checked(msg):
    if msg.stop_reason == "max_tokens":
        raise Truncated(f"cut off at max_tokens; {msg.usage.output_tokens} tokens produced")
    if msg.stop_reason == "refusal":
        raise RuntimeError(f"refused: {msg.stop_details}")
    return msg

try:
    checked(client.messages.create(model=MODEL, max_tokens=20, output_config={"effort": "low"},
                                   messages=[{"role": "user", "content": "List 50 animals, one per line."}]))
except Truncated as e:
    print("Truncated:", e)

Truncated: cut off at max_tokens; 20 tokens produced


### 3.8 Validation-retry: once, with exact feedback, append-only  [$]
Generic version: a code validator, the model's answer, the precise problems, one corrective turn, then fail loudly.

In [29]:
class Primes(BaseModel):
    numbers: list[int]

def is_prime(n): return n > 1 and all(n % d for d in range(2, int(n ** 0.5) + 1))

def validate(p: Primes) -> list[str]:
    problems = [f"{n} is not prime" for n in p.numbers if not is_prime(n)]
    if len(p.numbers) != 5:
        problems.append(f"expected 5 numbers, got {len(p.numbers)}")
    return problems

msgs = [{"role": "user", "content": "Give 5 two-digit odd numbers, all prime, as JSON."}]
for attempt in (1, 2):
    r = client.messages.parse(model=MODEL, max_tokens=1000, output_config={"effort": "low"},
                              output_format=Primes, messages=msgs)
    problems = validate(r.parsed_output)
    print(f"attempt {attempt}: {r.parsed_output.numbers} -> {problems or 'OK'}")
    if not problems:
        break
    msgs = msgs + [{"role": "assistant", "content": r.content},
                   {"role": "user", "content": "Validation failed:\n" + "\n".join(problems) + "\nReturn a corrected list."}]
else:
    raise RuntimeError(f"invalid after retry: {problems}")

attempt 1: [11, 13, 17, 19, 23] -> OK


### 3.9 Exit codes: one line, one code, no traceback
Scripts and CI branch on the code; humans read the line.

In [30]:
EXIT = {anthropic.AuthenticationError: 2, anthropic.RateLimitError: 3, anthropic.InternalServerError: 3,
        anthropic.APIConnectionError: 3, anthropic.APIStatusError: 5}

def exit_code(e: Exception) -> int:
    for cls, code in EXIT.items():                 # dict order = most specific first
        if isinstance(e, cls):
            return code
    return 1

bad = anthropic.Anthropic(api_key="sk-ant-api03-bad", max_retries=0)
try:
    bad.messages.create(model=MODEL, max_tokens=5, messages=[{"role": "user", "content": "hi"}])
except anthropic.APIError as e:
    print(f"{type(e).__name__}: exit {exit_code(e)}")

AuthenticationError: exit 2


## 4. Prompts as versioned files

### 4.1 Store, load, hash
Immutable files; label + content hash travel with every result.

In [31]:
PROMPTS = Path("prompts_demo"); PROMPTS.mkdir(exist_ok=True)
(PROMPTS / "qa.v1.md").write_text("Answer from the rules. Cite rule numbers.\n")
(PROMPTS / "qa.v2.md").write_text("Answer from the rules. Cite rule numbers. If no rule applies, say so.\n")

def load_prompt(name, version):
    text = (PROMPTS / f"{name}.{version}.md").read_text()
    return text, f"{name}.{version}", hashlib.sha256(text.encode()).hexdigest()[:8]

for v in ("v1", "v2"):
    print(load_prompt("qa", v)[1:])

('qa.v1', 'd89034d0')
('qa.v2', '2c1a5460')


### 4.2 Immutability check (the unit test)
A registry of released hashes; any in-place edit fails loudly.

In [32]:
RELEASED = {"qa.v1": load_prompt("qa", "v1")[2]}

def assert_immutable():
    for label, sha in RELEASED.items():
        name, version = label.rsplit(".", 1)
        now = load_prompt(name, version)[2]
        assert now == sha, f"{label} was edited in place: {sha} -> {now}; release a new version instead"

assert_immutable(); print("ok")
(PROMPTS / "qa.v1.md").write_text("Answer from the rules.\n")       # simulate an in-place edit
try:
    assert_immutable()
except AssertionError as e:
    print(e)
(PROMPTS / "qa.v1.md").write_text("Answer from the rules. Cite rule numbers.\n")   # restore

ok
qa.v1 was edited in place: d89034d0 -> b144b5c3; release a new version instead


42

### 4.3 A/B harness: same dataset, two versions, scored by code  [$]
The comparison only means something because each result is tagged with its version.

In [33]:
DATASET = [("Which team gets paged for service-3 error E3?", "team-3"),
           ("Which team gets paged for service-1 error E1?", "team-1"),
           ("Which team gets paged for service-9 error E99?", "none")]   # no such rule

def run_version(version):
    system, label, sha = load_prompt("qa", version)
    rows = []
    for q, expected in DATASET:
        r = client.messages.create(model=MODEL, max_tokens=300, output_config={"effort": "low"},
                                   system=system,
                                   messages=[{"role": "user", "content": [
                                       {"type": "text", "text": REFERENCE, "cache_control": {"type": "ephemeral"}},
                                       {"type": "text", "text": q}]}])
        answer = text_of(r).lower()
        rows.append({"prompt_version": label, "prompt_sha": sha, "q": q,
                     "correct": expected in answer or (expected == "none" and "no rule" in answer)})
    return rows

results = run_version("v1") + run_version("v2")
for label in ("qa.v1", "qa.v2"):
    rs = [r for r in results if r["prompt_version"] == label]
    print(label, f"{sum(r['correct'] for r in rs)}/{len(rs)}")

qa.v1 3/3
qa.v2 3/3


## 5. Batches API

### 5.1 `custom_id`: the only link between a request and its result
1-64 characters, `[a-zA-Z0-9_-]` only. Encode routing info; validate before submitting.

In [34]:
CID = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")
for cid in ["INC-001__filtered-v1", "INC-001|summarize.filtered.v1", "q_" + "x" * 70]:
    print(f"{cid[:40]:<42} valid={bool(CID.match(cid))}")

INC-001__filtered-v1                       valid=True
INC-001|summarize.filtered.v1              valid=False
q_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx   valid=False


### 5.2 Create: a list of `{custom_id, params}`; `params` is a normal Messages body  [$]
The response is a Message Batch object, not answers. Store its id.

In [35]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

questions = {f"q-{i:02d}": f"Which team gets paged for service-{i} error E{i}?" for i in range(1, 6)}
batch = client.messages.batches.create(requests=[
    Request(custom_id=cid,
            params=MessageCreateParamsNonStreaming(
                model=MODEL, max_tokens=300, output_config={"effort": "low"},
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": REFERENCE,
                     "cache_control": {"type": "ephemeral", "ttl": "1h"}},   # shared prefix: 1h, best-effort hits
                    {"type": "text", "text": q}]}]))
    for cid, q in questions.items()])
print(batch.id, batch.processing_status, batch.request_counts.model_dump())

msgbatch_011N3JAxztGEE4tTZ1kCbWwn in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}


### 5.3 Poll until `ended`  [free]

In [37]:
while (b := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    print(time.strftime("%H:%M:%S"), b.processing_status, b.request_counts.model_dump())
    time.sleep(30)
print("ended", b.request_counts.model_dump(), b.created_at, "->", b.ended_at)

14:50:16 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:50:46 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:51:17 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:51:47 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:52:18 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:52:48 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:53:18 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:53:49 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:54:19 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:54:50 in_progress {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 5, 'succeeded': 0}
14:55:20 in_progress

### 5.4 Results: JSONL, any order, four result types, route by `custom_id`  [free]

In [38]:
order = []
for res in client.messages.batches.results(batch.id):
    order.append(res.custom_id)
    match res.result.type:
        case "succeeded":
            m = res.result.message
            print(f"{res.custom_id}: {text_of(m)[:60]!r}  read={m.usage.cache_read_input_tokens}")
        case "errored":
            print(res.custom_id, "errored:", res.result.error.error.type)
        case "canceled" | "expired":
            print(res.custom_id, res.result.type, "-> resubmit")
print("submission order:", list(questions), "\nresult order:    ", order)

q-04: '**Team-0** gets paged for service-4 error E4. This is Rule 0'  read=2703
q-01: '**Team-1** gets paged. Rule 001 covers this case: when servi'  read=0
q-03: '**team-3** is paged. Rule 003 covers this case: when service'  read=2703
q-05: '**team-1** gets paged. Rule 005 covers this case: when servi'  read=2703
q-02: '**Team-2** gets paged. Rule 002 covers this case: when servi'  read=2703
submission order: ['q-01', 'q-02', 'q-03', 'q-04', 'q-05'] 
result order:     ['q-04', 'q-01', 'q-03', 'q-05', 'q-02']


### 5.5 Structured output in a batch: no `parse()` helper
Pass the JSON schema yourself (`transform_schema` builds what `parse()` would send); validate each result with Pydantic.

In [39]:
schema_params = MessageCreateParamsNonStreaming(
    model=MODEL, max_tokens=500,
    output_config={"effort": "low", "format": {"type": "json_schema", "schema": transform_schema(Answer)}},
    messages=[{"role": "user", "content": REFERENCE + "\nWhich rules page team-2?"}])
print(json.dumps(schema_params["output_config"]["format"]["schema"], indent=1)[:500])
# after collection:  Answer.model_validate_json(text_of(res.result.message))

{
 "type": "object",
 "title": "Answer",
 "properties": {
  "rule_numbers": {
   "type": "array",
   "title": "Rule Numbers",
   "items": {
    "type": "integer"
   }
  },
  "reasoning": {
   "type": "string",
   "title": "Reasoning"
  }
 },
 "additionalProperties": false,
 "required": [
  "rule_numbers",
  "reasoning"
 ]
}


### 5.6 List and cancel  [free]

In [40]:
for b in client.messages.batches.list(limit=5):
    print(b.id, b.processing_status, b.request_counts.model_dump())
# client.messages.batches.cancel(batch_id)  -> processing_status "canceling", then "ended" with partial results

msgbatch_011N3JAxztGEE4tTZ1kCbWwn ended {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 0, 'succeeded': 5}
msgbatch_011R9qjRReihfyZpkhPnrrSN ended {'canceled': 0, 'errored': 0, 'expired': 0, 'processing': 0, 'succeeded': 5}


## 6. Measurement

### 6.1 JSONL: append one record per call; read back as a table

In [41]:
LOG = Path("demo_runs.jsonl")
def log_call(**rec):
    with LOG.open("a") as f:
        f.write(json.dumps({"run_id": uuid.uuid4().hex[:8], **rec}) + "\n")

random.seed(1)
for version in ("qa.v1", "qa.v2"):
    for i in range(6):
        w = 1500 if i == 0 else 0
        log_call(prompt_version=version, cache_write=w, cache_read=0 if i == 0 else 1500,
                 input_tokens=20, output_tokens=random.randint(80, 200),
                 latency_s=round(random.uniform(1, 3), 2), correct=random.random() < (0.6 if version == "qa.v1" else 0.85))

import pandas as pd
df = pd.read_json(LOG, lines=True)
df.tail()

,run_id,prompt_version,cache_write,cache_read,input_tokens,output_tokens,latency_s,correct
7,08d40397,qa.v2,0,1500,20,93,2.80,True
8,7db8bee9,qa.v2,0,1500,20,83,2.30,True
9,c572503d,qa.v2,0,1500,20,192,1.76,True
10,d32719d0,qa.v2,0,1500,20,134,2.45,True
11,e763e039,qa.v2,0,1500,20,177,1.88,True


### 6.2 Aggregate: the comparison table

In [42]:
df["prompt_tokens"] = df.input_tokens + df.cache_write + df.cache_read
df["cost"] = (df.input_tokens * 0.10 + df.cache_write * 0.125 + df.cache_read * 0.01 + df.output_tokens * 0.50) / 1e6
table = df.groupby("prompt_version").agg(calls=("run_id", "count"),
                                         median_prompt=("prompt_tokens", "median"),
                                         cost=("cost", "sum"),
                                         p50_latency=("latency_s", "median"),
                                         accuracy=("correct", "mean"))
table["hit_rate"] = df.groupby("prompt_version").apply(
    lambda g: g.cache_read.sum() / (g.cache_read.sum() + g.cache_write.sum() + g.input_tokens.sum()))
table

,calls,median_prompt,cost,p50_latency,accuracy,hit_rate
prompt_version,,,,,,
qa.v1,6,1520.0,0.000671,1.645,0.666667,0.822368
qa.v2,6,1520.0,0.000700,2.375,1.000000,0.822368


### 6.3 Cost per completed task, not per call
A cheaper call that needs retries or fails more often is not cheaper. Divide total spend by correct results.

In [43]:
per_task = df.groupby("prompt_version").apply(lambda g: g.cost.sum() / max(g.correct.sum(), 1))
print((per_task * 1e6).round(1).astype(str) + " micro-$ per correct answer")
LOG.unlink()     # demo file only

prompt_version
qa.v1    167.6 micro-$ per correct answer
qa.v2    116.7 micro-$ per correct answer
dtype: str


#### END